# MNIST — обучение MLP (PyTorch)

Пайплайн: данные → модель → цикл обучения/теста → сохранение весов.

## 1. Импорты

In [ ]:
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

MNIST_MEAN, MNIST_STD = 0.1307, 0.3081


def load_mnist():
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((MNIST_MEAN,), (MNIST_STD,)),
    ])
    train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
    test_ds = datasets.MNIST(root="./data", train=False, download=True, transform=transform)
    return train_ds, test_ds

## 2. Модель

MLP: два скрытых слоя с BatchNorm + Dropout, финальный слой на 10 классов.

In [2]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.fc2 = nn.Linear(256, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.fc3 = nn.Linear(128, 10)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = x.flatten(1)
        x = F.relu(self.bn1(self.fc1(x)))
        x = self.dropout(x)
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.dropout(x)
        return self.fc3(x)

## 3. Функции обучения и оценки

In [3]:
def train(model, device, loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        loss = F.cross_entropy(model(data), target)
        loss.backward()
        optimizer.step()
        if batch_idx % 200 == 0:
            print(f"epoch {epoch} [{batch_idx * len(data)}/{len(loader.dataset)}] loss: {loss.item():.4f}")

In [4]:
def test(model, device, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            pred = model(data).argmax(dim=1)
            correct += pred.eq(target).sum().item()
    acc = correct / len(loader.dataset)
    print(f"test accuracy: {acc:.4f}")
    return acc

## 4. Устройство (CUDA / MPS / CPU)

In [5]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("using device:", device)

using device: mps


## 5. Данные

In [ ]:
train_ds, test_ds = load_mnist()

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256)

## 6. Модель и оптимизатор

In [7]:
model = MLP().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

## 7. Цикл обучения

In [8]:
total_start = time.time()
for epoch in range(1, 11):
    epoch_start = time.time()
    train(model, device, train_loader, optimizer, epoch)
    test(model, device, test_loader)
    print(f"epoch {epoch} took {time.time() - epoch_start:.2f}s")
print(f"total training time: {time.time() - total_start:.2f}s")

epoch 1 [0/60000] loss: 2.2882
epoch 1 [25600/60000] loss: 0.2480
epoch 1 [51200/60000] loss: 0.2843
test accuracy: 0.9681
epoch 1 took 4.64s
epoch 2 [0/60000] loss: 0.1821
epoch 2 [25600/60000] loss: 0.2444
epoch 2 [51200/60000] loss: 0.1292
test accuracy: 0.9749
epoch 2 took 3.36s
epoch 3 [0/60000] loss: 0.1577
epoch 3 [25600/60000] loss: 0.0936
epoch 3 [51200/60000] loss: 0.0930
test accuracy: 0.9757
epoch 3 took 3.54s
epoch 4 [0/60000] loss: 0.0811
epoch 4 [25600/60000] loss: 0.0581
epoch 4 [51200/60000] loss: 0.0917
test accuracy: 0.9773
epoch 4 took 3.63s
epoch 5 [0/60000] loss: 0.1186
epoch 5 [25600/60000] loss: 0.0534
epoch 5 [51200/60000] loss: 0.0838
test accuracy: 0.9775
epoch 5 took 3.38s
epoch 6 [0/60000] loss: 0.0415
epoch 6 [25600/60000] loss: 0.0534
epoch 6 [51200/60000] loss: 0.0618
test accuracy: 0.9821
epoch 6 took 3.61s
epoch 7 [0/60000] loss: 0.0873
epoch 7 [25600/60000] loss: 0.0502
epoch 7 [51200/60000] loss: 0.0317
test accuracy: 0.9818
epoch 7 took 3.59s
epoch 

## 8. Сохранение модели

In [9]:
torch.save(model.state_dict(), "mnist_mlp.pt")
print("model saved to mnist_mlp.pt")

model saved to mnist_mlp.pt


## 9. То же самое на TensorFlow (Keras)

Та же архитектура MLP, тот же датасет, идиоматичный Keras-стиль (`model.fit`).

In [18]:
import os
import time

import certifi

os.environ["SSL_CERT_FILE"] = certifi.where()

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("using device:", "GPU" if tf.config.list_physical_devices("GPU") else "CPU")

using device: CPU


In [14]:
MNIST_MEAN, MNIST_STD = 0.1307, 0.3081

(tf_train_x, tf_train_y), (tf_test_x, tf_test_y) = keras.datasets.mnist.load_data()

tf_train_x = (tf_train_x.astype("float32") / 255.0 - MNIST_MEAN) / MNIST_STD
tf_test_x = (tf_test_x.astype("float32") / 255.0 - MNIST_MEAN) / MNIST_STD

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [15]:
tf_model = keras.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(256),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.3),
    layers.Dense(128),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.3),
    layers.Dense(10),
])

tf_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

/Users/dashanalivayko/Library/Python/3.11/lib/python/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [16]:
class EpochTimer(keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.total_start = time.time()

    def on_epoch_begin(self, epoch, logs=None):
        self.epoch_start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        print(f"epoch {epoch + 1} took {time.time() - self.epoch_start:.2f}s")

    def on_train_end(self, logs=None):
        print(f"total training time: {time.time() - self.total_start:.2f}s")

In [17]:
tf_model.fit(
    tf_train_x, tf_train_y,
    validation_data=(tf_test_x, tf_test_y),
    batch_size=128,
    epochs=10,
    callbacks=[EpochTimer()],
)

Epoch 1/10
455/469 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8910 - loss: 0.3677epoch 1 took 1.79s
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8923 - loss: 0.3631 - val_accuracy: 0.9588 - val_loss: 0.1369
Epoch 2/10
463/469 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9480 - loss: 0.1730epoch 2 took 0.95s
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9481 - loss: 0.1730 - val_accuracy: 0.9717 - val_loss: 0.0956
Epoch 3/10
452/469 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9589 - loss: 0.1350epoch 3 took 1.10s
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9589 - loss: 0.1349 - val_accuracy: 0.9748 - val_loss: 0.0817
Epoch 4/10
449/469 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9659 - loss: 0.1120epoch 4 took 1.08s
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9661 - loss: 0.1109 - val_accuracy: 0.9767 - val_loss: 0.0743
Epoch 5/10
460/469 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9692 - loss: 0.0985epoch 5 took 1.17s
469/469 ━━━━

In [ ]:
tf_model.save("mnist_mlp_tf.keras")
print("model saved to mnist_mlp_tf.keras")

## 10. Сравнение времени: CPU vs GPU (PyTorch)

Та же модель и данные, обучение на CPU — для сравнения с временем на `device` из раздела 7 (там было `mps`).

In [19]:
cpu_device = torch.device("cpu")
cpu_model = MLP().to(cpu_device)
cpu_optimizer = torch.optim.Adam(cpu_model.parameters(), lr=1e-3)

cpu_total_start = time.time()
for epoch in range(1, 11):
    epoch_start = time.time()
    train(cpu_model, cpu_device, train_loader, cpu_optimizer, epoch)
    test(cpu_model, cpu_device, test_loader)
    print(f"epoch {epoch} took {time.time() - epoch_start:.2f}s")
cpu_total_time = time.time() - cpu_total_start
print(f"total training time (cpu): {cpu_total_time:.2f}s")

epoch 1 [0/60000] loss: 2.3922
epoch 1 [25600/60000] loss: 0.1979
epoch 1 [51200/60000] loss: 0.1533
test accuracy: 0.9648
epoch 1 took 3.73s
epoch 2 [0/60000] loss: 0.1326
epoch 2 [25600/60000] loss: 0.1265
epoch 2 [51200/60000] loss: 0.1001
test accuracy: 0.9731
epoch 2 took 3.59s
epoch 3 [0/60000] loss: 0.1510
epoch 3 [25600/60000] loss: 0.1247
epoch 3 [51200/60000] loss: 0.1086
test accuracy: 0.9759
epoch 3 took 3.21s
epoch 4 [0/60000] loss: 0.0372
epoch 4 [25600/60000] loss: 0.0498
epoch 4 [51200/60000] loss: 0.1076
test accuracy: 0.9788
epoch 4 took 2.88s
epoch 5 [0/60000] loss: 0.1224
epoch 5 [25600/60000] loss: 0.0779
epoch 5 [51200/60000] loss: 0.0422
test accuracy: 0.9797
epoch 5 took 2.76s
epoch 6 [0/60000] loss: 0.0356
epoch 6 [25600/60000] loss: 0.1344
epoch 6 [51200/60000] loss: 0.0634
test accuracy: 0.9805
epoch 6 took 2.82s
epoch 7 [0/60000] loss: 0.0820
epoch 7 [25600/60000] loss: 0.0380
epoch 7 [51200/60000] loss: 0.0916
test accuracy: 0.9821
epoch 7 took 2.85s
epoch 

In [20]:
gpu_total_time = 36.20  # из вывода раздела 7 (device: mps)

speedup = cpu_total_time / gpu_total_time
print(f"{device} time: {gpu_total_time:.2f}s")
print(f"cpu time: {cpu_total_time:.2f}s")
print(f"speedup: {speedup:.2f}x")

mps time: 36.20s
cpu time: 30.54s
speedup: 0.84x
